In [2]:
import re 
import pandas as pd 
import numpy as np
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
import torch 
from transformers import AutoTokenizer, Trainer, TrainingArguments, \
    BertForSequenceClassification, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
df = pd.read_json("../data/민원(콜센터) 질의응답_다산콜센터_일반행정 문의_Training.json")
df.info()

In [7]:
df.rename(
    columns = {
        '도메인' : ' 도메인'
    }, inplace = True
)

In [34]:
df.iloc[18: 30]

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
18,다산콜센터,일반행정 문의,B2240,고객,19,지방세납부,,Q,다른곳에서는 납부할수 없습니까?,,,,,납부방법/지방세/ 위텍스/ 납부,
19,다산콜센터,일반행정 문의,B2240,상담사,20,,지방세납부,A,,,,이용하시는 은행의 사이트에서도 지방세 납부가 가능합니다.,"은행, 사이트",은행/공공기관,"사이트,공공기관"
20,다산콜센터,일반행정 문의,B2241,고객,1,지방세납부,,Q,지방세는 조회할 수 있습니까?,,,,"지방세, 조회",지방세/세금,"조회,세금"
21,다산콜센터,일반행정 문의,B2241,상담사,2,,지방세납부,A,,,,간단한 본인확인 후 안내해드리겠습니다.,"본인확인, 안내",,안내
22,다산콜센터,일반행정 문의,B2241,고객,3,지방세납부,,A,,,알겠습니다.,,,,
23,다산콜센터,일반행정 문의,B2241,상담사,4,,지방세납부,Q,,지금 전화거신 핸드폰이 본인명의 맞습니까?,,,"핸드폰, 본인명의",핸드폰/전자기기,"본인명의,전자기기"
24,다산콜센터,일반행정 문의,B2241,고객,5,지방세납부,,A,,,맞습니다.,,,,
25,다산콜센터,일반행정 문의,B2241,상담사,6,,지방세납부,Q,,생년월일을 말씀해주시겠습니까?,,,생년월일,생년월일/개인정보,"생년월일,개인정보"
26,다산콜센터,일반행정 문의,B2241,고객,7,지방세납부,,A,,,OO월OO일 입니다.,,,,
27,다산콜센터,일반행정 문의,B2241,상담사,8,,지방세납부,Q,,살고계신 주소는 어디입니까?,,,주소,주소/개인정보,"주소,개인정보"


In [11]:
# 데이터프레임 로드 하고 컬럼의 이름들을 확인! 
df.columns.str.strip()

Index(['도메인', '카테고리', '대화셋일련번호', '화자', '문장번호', '고객의도', '상담사의도', 'QA',
       '고객질문(요청)', '상담사질문(요청)', '고객답변', '상담사답변', '개체명', '용어사전', '지식베이스'],
      dtype='str')

In [12]:
df.columns.map( lambda x : x.strip() )

Index(['도메인', '카테고리', '대화셋일련번호', '화자', '문장번호', '고객의도', '상담사의도', 'QA',
       '고객질문(요청)', '상담사질문(요청)', '고객답변', '상담사답변', '개체명', '용어사전', '지식베이스'],
      dtype='str')

In [14]:
df.columns = [ x.strip() for x in df.columns ]

In [ ]:
df.head()

In [39]:
# 필요한 컬럼을 제외하고 나머지는 제외 
df2 = df[['고객질문(요청)', '상담사답변']]

In [40]:
df2.rename(
    columns = {
        '고객질문(요청)' : '고객질문'
    }, inplace = True
)

In [41]:
# 텍스트 정규화 
def normalize(text):
    text = re.sub(r"[^가-힣0-9a-zA-Z\s\.]", ' ', str(text))
    text = re.sub(r"\s+", ' ', text).strip()

    return text

In [42]:
df2 = df2.map(normalize)

In [43]:
flag1 = (df2['고객질문'] != '') & (df2['상담사답변'].shift(-1) != '') & (df['문장번호'] == 1)
df3 = df2.loc[flag1, ]

In [44]:
# flag1을 한칸씩 밑으로 내리면 상담사의 답변
df3['상담사답변'] = df2.loc[flag1.shift(1).fillna(False), '상담사답변'].tolist()

In [45]:
df3.head()

,고객질문,상담사답변
0,지방세를 내려면 어떻게 해야됩니까,이용하시는 은행의 사이트에서 지방세 납부가 가능합니다.
20,지방세는 조회할 수 있습니까,간단한 본인확인 후 안내해드리겠습니다.
40,서울시주최 페스티벌 예매해놨는데 예정대로 진행됩니까,현재로썬 진행될 예정입니다.
60,청년저축계좌 지금 신청할 수 있습니까,죄송하지만 이미 신청기간이 지났습니다.
200,보건증 무인발급기로 출력할 수 있습니까,보건증은 모든 보건소에서 지원하는게 아니라서 검사받은 보건소 확인이 필요합니다.


In [46]:
df3.info()

<class 'pandas.DataFrame'>
Index: 1087 entries, 0 to 50316
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   고객질문    1087 non-null   str  
 1   상담사답변   1087 non-null   str  
dtypes: str(2)
memory usage: 149.2 KB


In [ ]:
# 기존의 데이터의 질문과 답변은 정상적인 답변 labels를 1로 채워준다. 
df3['labels'] = 1

In [48]:
# 기존의 질문은 유지한채 답변은 바꿔서 labels가 0인 구간을 생성 
answer_list = [
    ['A', 'a'], 
    ['B', 'b'], 
    ['C', 'c'], 
    ['D', 'd']
]
neg_list = []
for q, a in answer_list:
    # q : 질문
    # a : 답변

    cand = []
    for q2, a2 in answer_list:
        # a2 : 답변들의 목록
        if a != a2:
            cand.append(a2)
    # cand 틀린 답변의 목록에서 무작위로 하나를 선택 
    neg_a = np.random.choice(cand)
    neg_list.append([q, neg_a])

neg_list


[['A', np.str_('b')],
 ['B', np.str_('d')],
 ['C', np.str_('a')],
 ['D', np.str_('b')]]

In [54]:
answer_list2 = df3[['고객질문', '상담사답변']].values.tolist()

In [ ]:
neg_list2 = []
for q, a in answer_list2:
    cand = [ a2 for q2, a2 in answer_list2 if a != a2 ]

    neg_a = np.random.choice(cand)
    neg_list2.append( [q, neg_a] )
neg_list2

In [56]:
neg_df = pd.DataFrame(neg_list2, columns = ['고객질문', '상담사답변'])
neg_df['labels'] = 0
neg_df.head()

,고객질문,상담사답변,labels
0,지방세를 내려면 어떻게 해야됩니까,예 성실히 답변 드리겠습니다.,0
1,지방세는 조회할 수 있습니까,청년 전세임대주택은 입주대상자로 선정된 청년이 거주를 희망하는 주택을 고르면 LH가...,0
2,서울시주최 페스티벌 예매해놨는데 예정대로 진행됩니까,반려동물 행사가 있습니다,0
3,청년저축계좌 지금 신청할 수 있습니까,네 감사합니다. 불편한 점이나 문의사항이 있으신가요,0
4,보건증 무인발급기로 출력할 수 있습니까,네. OOOOO 임대아파트가 있습니다.,0


In [57]:
# df3와 neg_df을 단순 행 결합 
dataset_df = pd.concat(
    [df3.head(500), neg_df.head(500)], axis = 0, ignore_index=True
)

In [58]:
dataset_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   고객질문    1000 non-null   str  
 1   상담사답변   1000 non-null   str  
 2   labels  1000 non-null   int64
dtypes: int64(1), str(2)
memory usage: 134.6 KB


In [59]:
dataset_df['labels'].value_counts()

labels
1    500
0    500
Name: count, dtype: int64

In [60]:
train_df, test_df = train_test_split(
    dataset_df, test_size = 0.5, random_state=42, stratify= dataset_df['labels']
)

In [61]:
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))
# DatasetDict
ds = DatasetDict(
    {
        'train' : train_ds, 
        'validation' : test_ds
    }
)

In [63]:
ds

DatasetDict({
    train: Dataset({
        features: ['고객질문', '상담사답변', 'labels'],
        num_rows: 500
    })
    validation: Dataset({
        features: ['고객질문', '상담사답변', 'labels'],
        num_rows: 500
    })
})

In [64]:
MODEL_NAME = 'beomi/kcbert-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast = False)
max_len = 128

def token_fn(batch):
    # batch -> dict{'고객질문', '상담사답변'}
    tok = tokenizer(
        batch['고객질문'], 
        batch['상담사답변'], 
        truncation = True, 
        max_length = max_len
    )

    # 토큰화된 데이터에서 token_type_ids는 kobert 모델에서는 사용하지 않는다. 
    # 해당 키를 제거 
    tok.pop("token_type_ids", None)
    return tok

# remove_columns 매개변수 -> 토큰화를 하고 제외시킬 컬럼을 지정 
tok_ds = ds.map(
    token_fn, 
    batched = True, 
    remove_columns = [col for col in dataset_df.columns if col not in ['labels']], 
    # 캐시 사용 안함 
    load_from_cache_file = False
)

Map: 100%|██████████| 500/500 [00:00<00:00, 34891.47 examples/s]


In [ ]:
tok_ds['train'][0]['input_ids']

In [68]:
# 배치마다 동적으로 padding 토큰을 추가 
collator = DataCollatorWithPadding(
    tokenizer= tokenizer
)

In [69]:
# 학습 모델 정의 
# BertModel -> dropout -> Linear
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels = 2)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5116.10it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: beomi/kcbert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

In [70]:
# 평가 함수 
def metrics(eval_pred):
    logits, y = eval_pred
    pred = torch.argmax(logits, dim = 1)
    return {
        'accuracy_score' : accuracy_score(pred, y), 
        'f1_socre' : f1_score(pred, y)
    }

In [71]:
args = TrainingArguments(
    output_dir= '/model2', 
    eval_strategy='epoch', 
    save_strategy='epoch', 
    learning_rate=5e-05, 
    weight_decay=0.01, 
    warmup_steps=0.1, 
    num_train_epochs=3, 
    load_best_model_at_end=True, 
    metric_for_best_model='f1_score', 
    greater_is_better=True
)

In [72]:
trainer = Trainer(
    model = model, 
    args = args, 
    train_dataset= tok_ds['train'], 
    eval_dataset= tok_ds['validation'], 
    processing_class= tokenizer, 
    data_collator= collator, 
    compute_metrics= metrics
)

In [ ]:
trainer.train()

c:\Users\ekfla\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
